# Train Fontaine on Kaggle (free GPU)

Trains a Fontaine model on a Kaggle GPU and packages everything you need to
download (checkpoints + tokenizer) into `/kaggle/working`.

**Before running:** attach your data and turn on GPU —
- **Add Input → Upload → Dataset** with your raw corpus (`*.jsonl` / `*.txt`),
- Settings → **Accelerator: GPU T4 x2** (or P100),
- Settings → **Internet: On** (needed to clone the repo and pip install).

Full guide: `docs/kaggle.md` in the repo.

In [ ]:
# ---- Setup: edit these, then Run All ----
REPO_URL = "https://github.com/needyamin/yami.git"
BRANCH = "main"                      # or a specific commit/branch
REPO_DIR = "/kaggle/working/yami"

MODEL_CONFIG = "tiny"                # "tiny" | "small" | "medium"
RUN_NAME = "fontaine-kaggle"         # run dir: experiments/<timestamp>_<RUN_NAME>
MAX_STEPS = ""                       # e.g. "5000" to override kaggle.yaml; "" = keep config
SEQ_LEN = 256                        # data.sequence_length; must be <= model max (512 for small, 1024 for medium)

In [ ]:
import subprocess, sys

print("torch CUDA available:", end=" ")
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr or "nvidia-smi not found - enable GPU in notebook Settings -> Accelerator")

In [ ]:
# Clone the repo and install it (editable, with the fast hf_bpe tokenizer extra)
import os, subprocess, sys

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[bpe]"], check=True)

def fontaine(*args):
    """Run the fontaine CLI (module form works even when the console script is not on PATH)."""
    subprocess.run([sys.executable, "-m", "fontaine.cli.main", *args], check=True)

print("installed at:", REPO_DIR)

In [ ]:
# Copy raw data from every attached Kaggle Dataset into datasets/raw
import glob, os, shutil

RAW_DIR = os.path.join(REPO_DIR, "datasets", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

SKIP_NAMES = {"dataset-metadata.json", "kernel-metadata.json"}
copied = []
for path in sorted(glob.glob("/kaggle/input/**/*.*", recursive=True)):
    name = os.path.basename(path)
    if name.startswith(".") or name in SKIP_NAMES or name.endswith(".ipynb"):
        continue
    dest = os.path.join(RAW_DIR, name)
    if not os.path.exists(dest):     # keep original names; re-runs are no-ops
        shutil.copy2(path, dest)
    copied.append((name, os.path.getsize(dest)))

assert copied, "No data files found under /kaggle/input - attach your dataset (Add Input -> Upload)"
for name, size in copied:
    print(f"{size/1e6:8.2f} MB  {name}")

In [ ]:
# 1) Train the tokenizer on the corpus (byte-level BPE)
fontaine("tokenizer", "train",
         "--data-config", "configs/data/default.yaml",
         "--tokenizer-dir", "datasets/tokenizer",
         "--set", 'data.raw_paths=["datasets/raw"]',
         "--set", "tokenizer.type=hf_bpe")

In [ ]:
# 2) Validate + prepare token shards and the manifest
fontaine("data", "validate",
         "--data-config", "configs/data/default.yaml",
         "--set", 'data.raw_paths=["datasets/raw"]')

fontaine("data", "prepare",
         "--data-config", "configs/data/default.yaml",
         "--tokenizer-dir", "datasets/tokenizer",
         "--set", 'data.raw_paths=["datasets/raw"]',
         "--set", f"data.sequence_length={SEQ_LEN}",
         "--license", "CC-BY-4.0")

In [ ]:
# 3) Train (uses configs/training/kaggle.yaml: device/precision auto -> CUDA + AMP)
args = ["train",
        "--model-config", f"configs/model/{MODEL_CONFIG}.yaml",
        "--training-config", "configs/training/kaggle.yaml",
        "--data-config", "configs/data/default.yaml",
        "--tokenizer-dir", "datasets/tokenizer",
        "--set", "data.manifest_path=datasets/prepared/manifest.json",
        "--set", f"training.run_name={RUN_NAME}"]
if MAX_STEPS:
    args += ["--set", f"training.max_steps={MAX_STEPS}"]
fontaine(*args)

In [ ]:
# 4) Sanity check: generate a few tokens from the fresh checkpoint
import os

run_dirs = sorted(d for d in glob.glob("experiments/*") if RUN_NAME in os.path.basename(d))
assert run_dirs, "no experiments run directory found"
run_dir = run_dirs[-1]              # newest
print("run directory:", run_dir)

fontaine("generate",
         "--checkpoint", os.path.join(run_dir, "checkpoints"),
         "--tokenizer-dir", "datasets/tokenizer",
         "--prompt", "def fibonacci(n):",
         "--max-new-tokens", "64")

In [ ]:
# 5) Package checkpoints + tokenizer for download, then stop the GPU session
import glob, os, shutil, zipfile

def add_to_zip(zf, root):
    for path in glob.glob(os.path.join(root, "**", "*"), recursive=True):
        if os.path.isfile(path):
            zf.write(path, os.path.relpath(path, REPO_DIR))

zip_path = f"/kaggle/working/fontaine_{os.path.basename(run_dir)}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    add_to_zip(zf, run_dir)                # checkpoints + summary
    add_to_zip(zf, "datasets/tokenizer")   # tokenizer files

print(f"created {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print()
print("Download it from the Output panel on the right (or the Output tab of the")
print("saved version), then on your machine:")
print(f"  unzip {os.path.basename(zip_path)} -d <your-yami-clone>")
print("  fontaine generate --checkpoint <unzipped-run>/checkpoints \")
print("      --tokenizer-dir <unzipped>/datasets/tokenizer --interactive")